# Reinforcement Learning with Verifiable Rewards (RLVR)

In Session 15 we changed a model's weights with GRPO, rewarding it for verifiably correct answers. This session zooms into the other half of that loop — the **verifier** — and the data pipeline built around it. No GPU required: we run the RLVR sampling-and-verification loop against an API model, so the focus stays on the part that makes or breaks reinforcement learning on language models: the reward signal itself.

The RLVR loop looks like this:

```text
prompt -> sample N completions -> verify each against a deterministic checker
       -> assign rewards -> keep verified-correct samples as preference data
       -> policy update -> repeat
```

Unlike RLHF, there is no learned reward model and no human labeler in the loop. The reward comes from a *deterministic program* — a math answer checker, a unit-test runner — that either passes a completion or doesn't. That makes the signal cheap, objective, and reproducible. It also makes it a target: any policy trained against a verifier will find and exploit its blind spots, so we will also build reward-hacking detection and an audit trail that records every verifier decision.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain how RLVR differs from RLHF, and what makes a reward "verifiable."
- Implement verifiable reward functions for math (exact-answer matching) and code (unit-test execution).
- Run the sample-and-verify loop and interpret group-level accuracy.
- Detect reward-hacking signatures and maintain a verifier audit trail.
- Construct chosen/rejected preference pairs ready for DPO-style training — or reward signals ready for GRPO.

## Table of Contents

- **Breakout Room #1: The Verifiable Reward Loop**
  - Task 1: Environment Setup
  - Task 2: Problems and Answer Extraction
  - Task 3: A Math Reward Function
  - Question #1 and Question #2
  - Task 4: Sample and Verify
- **Breakout Room #2: Reward Hacking, Code Verification, and Preference Data**
  - Task 5: Reward-Hacking Detection and the Audit Trail
  - Question #3
  - Task 6: A Code Verifier
  - Question #4
  - Task 7: Build Preference Pairs
  - Activity #1
- **Conclusion: What We Built, Start to Finish**
- **What Looks Different in Production**

---
# Breakout Room #1
## The Verifiable Reward Loop

We build the core RLVR machinery: a small set of math problems with known answers, a deterministic answer checker, a reward function, and the sample-and-verify loop that turns them into training signal.

## Task 1: Environment Setup

From the `16_RLVR` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

You will need an [OpenAI API key](https://platform.openai.com/api-keys). Enter it below — it is kept in memory for this session only.

In [1]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")
else:
    print("OPENAI_API_KEY is already available in this kernel.")

OPENAI_API_KEY is already available in this kernel.


### The Policy

In RL terms, the model we sample from is the **policy**. We deliberately use a small model (`gpt-4.1-nano`) at temperature 1.0: a policy that is *sometimes wrong* is exactly what we want, because the contrast between verified-correct and verified-incorrect samples is where the training signal lives. A policy that never fails produces no gradient — and no preference pairs.

In [2]:
from openai import OpenAI

client = OpenAI(timeout=60.0, max_retries=1)

MODEL = "gpt-4.1-nano"  # small on purpose: we *want* some wrong answers


def sample_completions(
    prompt: str, system: str = "", temperature: float = 1.0, n: int = 1
) -> list[str]:
    """Sample one API batch from the policy model with bounded output."""
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        n=n,
        max_completion_tokens=512,
    )
    return [choice.message.content or "" for choice in response.choices]


def simple_complete(prompt: str, system: str = "", temperature: float = 1.0) -> str:
    """Return one completion for the later single-sample activities."""
    return sample_completions(prompt, system, temperature, n=1)[0]


print(simple_complete("Reply with exactly: policy online"))

policy online


## Task 2: Problems and Answer Extraction

RLVR only works in domains where correctness can be *checked by a program*. Math word problems with a single numeric answer are the canonical example (this is why GSM8K shows up in every RLVR paper — and in Session 15).

Two conventions make checking reliable:

1. Each problem carries a **ground-truth answer** as a string.
2. The prompt instructs the policy to put its final answer in `\boxed{}` — the same convention GSM8K-style training uses — so extraction is a regex, not a judgment call. If no box is found, we fall back to the last number in the text.

In [3]:
import re
from dataclasses import dataclass


@dataclass
class Problem:
    question: str
    answer: str  # ground truth


problems = [
    Problem("What is 12 * 13?", "156"),
    Problem("If a train covers 60 km in 45 minutes, what is its speed in km/h?", "80"),
    Problem("What is the sum of the first 10 positive integers?", "55"),
    Problem(
        "A store discounts a $250 jacket by 20%, then adds 10% sales tax on the discounted price. "
        "What is the final price in dollars?",
        "220",
    ),
    Problem("How many positive divisors does 360 have?", "24"),
    Problem("What is 12,347 multiplied by 8,911?", "110024117"),
    Problem("What is the least non-negative remainder when 17^123 is divided by 10,000?", "9713"),
    Problem("How many onto functions are there from a 6-element set to a 3-element set?", "540"),
]


NUMBER_PATTERN = re.compile(r"-?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?(?:[eE][+-]?\d+)?")


def boxed_contents(text: str) -> list[str]:
    """Return balanced contents from every LaTeX boxed expression."""
    contents = []
    marker = r"\boxed{"
    start = 0
    while (marker_index := text.find(marker, start)) != -1:
        content_start = marker_index + len(marker)
        depth = 1
        for index in range(content_start, len(text)):
            if text[index] == "{":
                depth += 1
            elif text[index] == "}":
                depth -= 1
                if depth == 0:
                    contents.append(text[content_start:index])
                    start = index + 1
                    break
        else:
            break
    return contents


def extract_number(text: str) -> str:
    r"""Extract one numeric value from one box; otherwise use the last number."""
    boxes = boxed_contents(text)
    if boxes:
        if len(boxes) != 1:
            return ""
        numbers = NUMBER_PATTERN.findall(boxes[0])
        return numbers[0].replace(",", "") if len(numbers) == 1 else ""
    numbers = NUMBER_PATTERN.findall(text)
    return numbers[-1].replace(",", "") if numbers else ""


assert extract_number(r"Step 1: 12*13 = 156. The answer is \boxed{156}.") == "156"
assert extract_number("So the speed is 80 km/h") == "80"
assert extract_number(r"The speed is \boxed{80\ \text{km/h}}") == "80"
assert extract_number(r"Conflicting: \boxed{80} then \boxed{81}") == ""
assert extract_number(r"The total is \boxed{1,234.5}") == "1234.5"
assert extract_number("no numbers here") == ""
print("extraction OK")

extraction OK


## Task 3: A Math Reward Function

The reward function is where verification becomes training signal:

- **+1.0** if the extracted answer matches the ground truth,
- **−0.1** otherwise.

Note the asymmetry: full credit for success, only a *small* penalty for failure. We also normalize numerically (`"80"` and `"80.0"` should match) — a verifier that fails on formatting technicalities punishes correct reasoning, which is the fastest way to teach a policy the wrong lesson.

In [4]:
class MathRewardFunction:
    """Verifiable reward for math problems: exact answer match against ground truth."""

    correct_reward = 1.0
    incorrect_penalty = -0.1

    def compute(self, response: str, ground_truth: str) -> float:
        prediction = extract_number(response)
        if self._normalize(prediction) == self._normalize(ground_truth):
            return self.correct_reward
        return self.incorrect_penalty

    @staticmethod
    def _normalize(value: str):
        """Compare numerically when possible, so '80' == '80.0'."""
        try:
            return float(value)
        except ValueError:
            return value.strip()


reward_fn = MathRewardFunction()

assert reward_fn.compute(r"The speed is \boxed{80}", "80.0") == 1.0
assert reward_fn.compute(r"The speed is \boxed{81}", "80") == -0.1
assert reward_fn.compute("I cannot solve this.", "80") == -0.1
print("reward function OK")

reward function OK


#### ❓ Question #1

In your own words: what makes a reward "verifiable," and how does RLVR differ from RLHF's learned reward model? Give one task where a verifiable reward exists and one where it fundamentally cannot (and explain why).

##### Answer:

A reward is verifiable when a deterministic program can decide whether the output satisfies an explicit contract and produce the same result every time. RLVR uses that programmatic result directly as the reward signal, whereas RLHF learns an approximate reward model from human preference labels and therefore reflects subjective judgments and possible model error. A Python solution is verifiable when it must pass a fixed unit-test suite. By contrast, whether a short story is emotionally moving has no single ground-truth checker: readers can reasonably disagree, so it needs human preferences or another subjective proxy rather than RLVR.

#### ❓ Question #2

The reward is asymmetric: +1.0 for a correct answer but only −0.1 for an incorrect one. Suppose we used −1.0 instead. What behavior might a policy learn during early training, when most of its attempts fail? (Hint: think about a model that discovers it can hedge, refuse, or produce no parseable answer at all.)

##### Answer:

A large negative reward on almost every early sample can make exploration unattractive. If refusals, hedged answers, or unparsable outputs are skipped, penalized less elsewhere in the stack, or simply cost fewer tokens while receiving the same failure reward, the policy can collapse toward those cheap defensive behaviors instead of attempting the task. Strictly in this notebook, every wrong or empty answer receives the same penalty, so changing -0.1 to -1.0 does not by itself make refusal better than another wrong answer; the risk appears through that interaction with KL, length/format rewards, filtering, or missing-answer handling. The smaller penalty preserves a strong success signal while making failed attempts comparatively cheap.

## Task 4: Sample and Verify

Now the heart of RLVR: for each problem, sample a **group** of completions at temperature 1.0 and verify every one.

Sampling groups (rather than one completion per prompt) is not incidental — it is the same structure GRPO consumed in Session 15, where each completion's advantage was computed *relative to its group's average reward*. Here the group serves a second purpose too: correct and incorrect completions of the same prompt become the raw material for preference pairs in Breakout Room #2.

In [5]:
from dataclasses import asdict


@dataclass
class Sample:
    problem: str
    response: str
    extracted: str
    reward: float
    verified_correct: bool


MATH_SYSTEM = "You are a careful mathematician. Show concise work, then box only the number."


def math_prompt(question: str) -> str:
    """Build the exact user prompt shared by sampling and preference data."""
    return (
        "Solve concisely in at most 8 short lines. Put only the final numeric answer "
        f"in \\boxed{{}}.\n\n{question}"
    )


def sample_and_verify(problem: Problem, n_samples: int = 4) -> list[Sample]:
    """Sample a group of completions for one problem and verify each one."""
    prompt = math_prompt(problem.question)
    responses = sample_completions(
        prompt,
        system=MATH_SYSTEM,
        temperature=1.3,
        n=n_samples,
    )
    group = []
    for response in responses:
        extracted = extract_number(response)
        reward = reward_fn.compute(response, problem.answer)
        group.append(
            Sample(
                problem=problem.question,
                response=response,
                extracted=extracted,
                reward=reward,
                verified_correct=reward > 0,
            )
        )
    return group


groups = [
    sample_and_verify(problem, n_samples=8 if index >= 5 else 4)
    for index, problem in enumerate(problems)
]

total = sum(len(g) for g in groups)
correct = sum(s.verified_correct for g in groups for s in g)
print(f"Verified-correct rate: {correct / total:.0%} ({correct}/{total})\n")
for problem, group in zip(problems, groups):
    print(f"  {sum(s.verified_correct for s in group)}/{len(group)}  {problem.question[:70]}")

Verified-correct rate: 70% (31/44)

  4/4  What is 12 * 13?
  4/4  If a train covers 60 km in 45 minutes, what is its speed in km/h?
  4/4  What is the sum of the first 10 positive integers?
  4/4  A store discounts a $250 jacket by 20%, then adds 10% sales tax on the
  4/4  How many positive divisors does 360 have?
  3/8  What is 12,347 multiplied by 8,911?
  0/8  What is the least non-negative remainder when 17^123 is divided by 10,
  8/8  How many onto functions are there from a 6-element set to a 3-element 


> NOTE: If your verified-correct rate is 100%, the contrast that drives learning is missing — swap in a smaller model, raise the temperature, or add harder problems until some samples fail.

Let's inspect one failure — reading verifier-rejected completions is how you learn what your policy actually gets wrong (arithmetic slips? misread units? unparseable formatting?):

In [6]:
incorrect = [s for g in groups for s in g if not s.verified_correct]
if incorrect:
    sample = incorrect[0]
    print(f"Problem:   {sample.problem}")
    print(f"Extracted: {sample.extracted!r}  (ground truth mismatch, reward {sample.reward})\n")
    print(sample.response)
else:
    print("No incorrect samples this run - try a harder problem or higher temperature.")

Problem:   What is 12,347 multiplied by 8,911?
Extracted: '109994117'  (ground truth mismatch, reward -0.1)

12437 * 8911 = (12000 + 347) * (8900 + 11)

= 12000 * 8900 + 12000 * 11 + 347 * 8900 + 347 * 11

= 106,800,000 + 132,000 + 3,078,300 + 3,817

Sum: 106,800,000 + 132,000 + 3,078,300 + 3,817 = 109,994,117

\boxed{109994117}


## Breakout Room #1 Summary

- Verifiable rewards come from deterministic checkers, not human preference or a learned reward model — cheap, objective, reproducible.
- The `\boxed{}` convention plus numeric normalization makes extraction mechanical; a brittle verifier punishes correct reasoning and corrupts the signal.
- Asymmetric rewards (+1.0 / −0.1) keep early training from collapsing into refusal.
- Sampling *groups* of completions per prompt is the same structure GRPO trains on — and it produces the correct/incorrect contrast that preference data needs.

---
# Breakout Room #2
## Reward Hacking, Code Verification, and Preference Data

A verifier is not just a metric — once you train against it, it *is* the objective. This room covers what happens then: policies that exploit the verifier's blind spots, a second verifiable domain (code judged by unit tests), and turning audited verifier output into preference data.

## Task 5: Reward-Hacking Detection and the Audit Trail

Goodhart's law — *"when a measure becomes a target, it ceases to be a good measure"* — is the central operational risk of RLVR. A policy optimized against an exact-match verifier will happily learn to:

- emit a bare boxed answer with no reasoning (guessing is cheap when only the box is checked),
- parrot numbers that appear in the prompt,
- or exploit extraction quirks instead of solving the problem.

Two defenses, both borrowed from how production RLVR systems are reviewed:

1. **Flag hack signatures.** Here we flag verified-correct samples that show *no visible work* — a right answer without reasoning is the classic signature of guessing or leakage.
2. **Log every verifier decision** to an append-only audit trail (`artifacts/verifier.jsonl`). When someone — a teammate, an auditor, a regulator — asks whether your RL run was trained on honest rewards, this file is the answer.

In [7]:
import json
from pathlib import Path

AUDIT_LOG = Path("artifacts/verifier.jsonl")
AUDIT_LOG.parent.mkdir(exist_ok=True)


def looks_like_hack(sample: Sample) -> bool:
    """Flag verified-correct samples that show no work.

    A correct boxed answer with no visible reasoning is the classic hack
    signature: the policy may be guessing, pattern-matching the prompt, or
    exploiting the extractor rather than solving the problem.
    """
    if not sample.verified_correct:
        return False
    work = sample.response.replace(f"\\boxed{{{sample.extracted}}}", "")
    numbers_in_work = re.findall(r"-?\d+\.?\d*", work)
    return len(sample.response.split()) < 20 or len(numbers_in_work) < 2


def audit_record(sample: Sample) -> dict:
    """Append one verifier decision to the audit trail and return it."""
    record = {**asdict(sample), "suspected_hack": looks_like_hack(sample)}
    with AUDIT_LOG.open("a") as f:
        f.write(json.dumps(record) + "\n")
    return record


# Keep one reproducible ledger per notebook run; records are append-only within it.
AUDIT_LOG.write_text("", encoding="utf-8")
records = [audit_record(s) for g in groups for s in g]
flagged = sum(r["suspected_hack"] for r in records)

print(f"Audited {len(records)} samples -> {flagged} flagged as hack-suspect")
print(f"Audit trail: {AUDIT_LOG} ({sum(1 for _ in AUDIT_LOG.open())} records total)")

Audited 44 samples -> 0 flagged as hack-suspect
Audit trail: artifacts\verifier.jsonl (44 records total)


#### ❓ Question #3

Our detector flags one signature: "right answer, no visible work." Name **two other ways** a policy could hack a `\boxed{}` exact-match verifier, and for each, describe how you would harden the verifier or the prompt against it. (Session 15's stacked format rewards are one relevant hardening example.)

##### Answer:

1. **Answer leakage or number parroting:** the policy can copy a number from the prompt, a worked example, or contaminated training data and place it in `\boxed{}` without solving anything. I would use held-out and procedurally randomized problems, remove answer-bearing examples from the prompt, and independently recompute the result with a symbolic checker.

2. **Parser exploitation:** the policy can emit multiple or ambiguous boxed values so the regex scores one value while a human or downstream consumer reads another. I would require exactly one terminal `\boxed{}` value matching a strict grammar, reject extra answer markers or trailing content, and keep format/process checks separate from correctness reward.

## Task 6: A Code Verifier

Math is one verifiable domain; **code judged by unit tests** is the other workhorse of RLVR. The verifier executes a candidate program against test cases and returns the *fraction that pass* — a graded reward in `[0.0, 1.0]` rather than math's binary match.

We run candidates in a subprocess with a timeout: a program that crashes, hangs, or exits non-zero simply earns no credit for that test case.

> ⚠️ We are executing model-generated code on your machine. For this demo the programs are trivial, but note the design: in production, this verifier runs inside a **sandbox** (container, gVisor, firecracker VM) — never on the host.

In [8]:
import subprocess
import sys
import tempfile


class CandidateProgramError(RuntimeError):
    """A candidate failed normally and should receive no credit."""


class CodeVerifier:
    """Score generated code by the fraction of test cases it passes."""

    timeout_seconds = 5

    def verify(self, code: str, test_cases: list[dict]) -> float:
        passed = 0
        for tc in test_cases:
            try:
                output = self._run(code, tc.get("input", ""))
                if output.strip() == str(tc["expected"]).strip():
                    passed += 1
            except (CandidateProgramError, subprocess.TimeoutExpired):
                continue  # candidate failure -> no credit; infrastructure errors still surface
        return passed / len(test_cases) if test_cases else 0.0

    def _run(self, code: str, input_data: str = "") -> str:
        # This reduces accidental exposure; it is still not a security sandbox.
        child_env = {
            key: os.environ[key]
            for key in ("SYSTEMROOT", "WINDIR", "PATH", "PATHEXT")
            if key in os.environ
        }
        with tempfile.TemporaryDirectory(prefix="rlvr-verifier-") as workdir:
            result = subprocess.run(
                [sys.executable, "-I", "-S", "-c", code],
                input=input_data,
                capture_output=True,
                text=True,
                timeout=self.timeout_seconds,
                cwd=workdir,
                env=child_env,
            )
        if result.returncode != 0:
            raise CandidateProgramError(result.stderr)
        return result.stdout


verifier = CodeVerifier()

# Sanity check with hand-written candidates: one correct, one buggy.
tests = [{"input": "3", "expected": "14"}, {"input": "10", "expected": "385"}]
good = "n = int(input()); print(sum(i * i for i in range(1, n + 1)))"
bad = "n = int(input()); print(sum(range(1, n + 1)))"  # sums i, not i^2

assert verifier.verify(good, tests) == 1.0
assert verifier.verify(bad, tests) == 0.0
print("code verifier OK")

code verifier OK


Now close the loop: have the **policy** write the program, and let the verifier score it — the exact reward signal a coding-RLVR run trains on.

In [9]:
CODING_TASK = (
    "Write a Python program that reads a single integer n from standard input "
    "and prints the sum of the squares of the integers from 1 to n (inclusive). "
    "Print only the number. Reply with only the code - no markdown fences, no explanation."
)

code_tests = [
    {"input": "1", "expected": "1"},
    {"input": "3", "expected": "14"},
    {"input": "10", "expected": "385"},
]


def strip_fences(text: str) -> str:
    """Remove markdown code fences if the policy ignores instructions."""
    return re.sub(r"^```(?:python)?\s*\n|\n?```\s*$", "", text.strip())


for i in range(3):
    candidate = strip_fences(simple_complete(CODING_TASK))
    score = verifier.verify(candidate, code_tests)
    print(f"candidate {i + 1}: reward = {score:.2f}")
    print("  " + candidate.replace("\n", "\n  ") + "\n")

candidate 1: reward = 1.00
  n = int(input())
  print(sum(i*i for i in range(1, n+1)))



candidate 2: reward = 1.00
  n = int(input())
  print(sum(i*i for i in range(1, n+1)))



candidate 3: reward = 1.00
  n = int(input())
  print(sum(i**2 for i in range(1, n+1)))



#### ❓ Question #4

The code verifier returns *fractional* rewards (fraction of tests passed) while the math verifier is binary. What are the benefits and risks of partial credit as a training signal? And concretely: what could a policy-generated program do to a verifier that runs candidates directly on the host, and which parts of that threat does our timeout **not** cover?

##### Answer:

Fractional rewards provide a denser signal: they distinguish a nearly correct program from a completely wrong one and can make learning more sample-efficient. The risks are that easy or duplicated tests dominate the score, the policy hard-codes visible cases, or it optimizes a subset of tests instead of learning the intended algorithm. Hidden tests, balanced test families, mutation tests, and a final all-or-nothing correctness gate help.

A host-executed candidate can read or modify files, steal environment variables and API keys, send them over the network, launch child processes, or exhaust CPU, memory, disk, and output buffers. The timeout only bounds how long the parent waits for the direct process. It does not block filesystem or network access, prevent exfiltration that finishes before the deadline, impose resource limits, or reliably terminate descendants. Production verification therefore needs a disposable sandbox with a minimal environment, network policy, resource limits, and teardown.

## Task 7: Build Preference Pairs

Finally, we turn audited verifier output into training data. Within each group:

- **chosen** = verified-correct samples that were *not* flagged as hack-suspect,
- **rejected** = verified-incorrect samples,

and we take the cross product. The resulting `{prompt, chosen, rejected}` records are exactly the format [DPO-style trainers](https://huggingface.co/docs/trl/dpo_trainer) consume — while GRPO (Session 15) skips the pairing and uses the group rewards directly. Same verifier, two consumers.

Excluding flagged samples matters: a hack-suspect completion used as "chosen" would teach the next policy iteration to hack *more*.

In [10]:
def build_preferences(groups: list[list[Sample]], records: list[dict]) -> list[dict]:
    """Cross verified-correct (unflagged) winners with incorrect losers, per group."""
    flagged_responses = {r["response"] for r in records if r["suspected_hack"]}
    pairs = []
    for group in groups:
        winners = [s for s in group if s.verified_correct and s.response not in flagged_responses]
        losers = [s for s in group if not s.verified_correct]
        pairs.extend(
            {
                "prompt": f"System: {MATH_SYSTEM}\nUser: {math_prompt(winner.problem)}",
                "chosen": winner.response,
                "rejected": loser.response,
            }
            for winner in winners
            for loser in losers
        )
    return pairs


pairs = build_preferences(groups, records)

PREFERENCES = Path("artifacts/preferences.jsonl")
with PREFERENCES.open("w") as f:
    for pair in pairs:
        f.write(json.dumps(pair) + "\n")

print(f"{len(pairs)} preference pairs -> {PREFERENCES}\n")
if pairs:
    example = pairs[0]
    print(f"prompt:   {example['prompt']}")
    print(f"chosen:   {example['chosen'][:120]}...")
    print(f"rejected: {example['rejected'][:120]}...")
else:
    print("No pairs this run - you need at least one correct AND one incorrect sample in the same group.")

15 preference pairs -> artifacts\preferences.jsonl

prompt:   System: You are a careful mathematician. Show concise work, then box only the number.
User: Solve concisely in at most 8 short lines. Put only the final numeric answer in \boxed{}.

What is 12,347 multiplied by 8,911?
chosen:   First, write the problem as \( 12,347 \times 8,911 \).

Use standard multiplication (or partial products):

- \( 12347 \...
rejected: 12437 * 8911 = (12000 + 347) * (8900 + 11)

= 12000 * 8900 + 12000 * 11 + 347 * 8900 + 347 * 11

= 106,800,000 + 132,000...


#### 🏗️ Activity #1: Build Your Own Verifier

Math answers and unit tests are only two verifiable domains. Pick another — for example:

- **JSON schema conformance**: does the completion parse and validate against a schema?
- **SQL correctness**: does a generated query return the same rows as a reference query on a fixture database?
- **Regex/string transformation**: does the output match a deterministic expected transformation of the input?

Then, in the cell below:

1. Implement a reward function for your domain (binary or fractional — justify the choice).
2. Run the sample-and-verify loop over at least 3 prompts with `n_samples >= 3`.
3. Report the verified-correct rate, and note any hack-suspect behavior you observe (and how you'd detect it).

In [11]:
# Domain: strict JSON conformance plus deterministic string normalization.
# Fractional reward exposes which part of the contract failed while 1.0 still
# requires the complete schema and the recomputed semantic values to match.

@dataclass(frozen=True)
class JsonContractTask:
    source_text: str
    expected: dict[str, object]


JSON_KEYS = {"slug", "length", "contains_digit"}


def expected_json_record(source_text: str) -> dict[str, object]:
    """Compute ground truth independently from the policy."""
    slug = re.sub(r"[^a-z0-9]+", "-", source_text.lower()).strip("-")
    return {
        "slug": slug,
        "length": len(slug),
        "contains_digit": any(char.isdigit() for char in slug),
    }


def strict_json_object(text: str) -> dict[str, object]:
    """Parse one JSON object and reject duplicate keys."""
    def reject_duplicates(pairs: list[tuple[str, object]]) -> dict[str, object]:
        result: dict[str, object] = {}
        for key, value in pairs:
            if key in result:
                raise ValueError(f"duplicate key: {key}")
            result[key] = value
        return result

    parsed = json.loads(text, object_pairs_hook=reject_duplicates)
    if not isinstance(parsed, dict):
        raise ValueError("top-level JSON value must be an object")
    return parsed


def json_contract_reward(response: str, expected: dict[str, object]) -> float:
    """Return the fraction of five independently verifiable checks passed."""
    try:
        value = strict_json_object(response.strip())
    except (json.JSONDecodeError, ValueError):
        return 0.0

    type_check = (
        isinstance(value.get("slug"), str)
        and type(value.get("length")) is int
        and type(value.get("contains_digit")) is bool
    )
    checks = [
        set(value) == JSON_KEYS,
        type_check,
        value.get("slug") == expected["slug"],
        value.get("length") == expected["length"],
        value.get("contains_digit") == expected["contains_digit"],
    ]
    return sum(checks) / len(checks)


def json_hack_signatures(response: str, expected: dict[str, object]) -> list[str]:
    """Flag surface-level compliance that does not satisfy the real contract."""
    try:
        value = strict_json_object(response.strip())
    except (json.JSONDecodeError, ValueError):
        return ["not strict JSON (prose, fences, or duplicate keys)"]
    flags = []
    if set(value) != JSON_KEYS:
        flags.append("missing or extra keys")
    if set(value) == JSON_KEYS and json_contract_reward(response, expected) < 1.0:
        flags.append("schema-shaped output with incorrect derived values")
    return flags


sources = [
    "Hello RLVR 16",
    "  Guardrails & Audit Trails!  ",
    "DPO__Preference   Pairs",
]
json_tasks = [JsonContractTask(source, expected_json_record(source)) for source in sources]

JSON_INSTRUCTIONS = """Normalize the source text using these exact rules:
1. Lowercase ASCII letters.
2. Replace every maximal run of non-ASCII-alphanumeric characters with one hyphen.
3. Remove leading and trailing hyphens.
Return exactly one JSON object with exactly these keys: slug (string), length
(number of characters in slug, integer), and contains_digit (boolean). No prose
or markdown fences.
"""


json_results = []
for task in json_tasks:
    for _ in range(3):
        response = simple_complete(
            f"{JSON_INSTRUCTIONS}\nSource text: {task.source_text!r}",
            system="Follow the output contract exactly.",
            temperature=1.0,
        )
        reward = json_contract_reward(response, task.expected)
        json_results.append(
            {
                "source": task.source_text,
                "response": response,
                "reward": reward,
                "verified_correct": reward == 1.0,
                "hack_signatures": json_hack_signatures(response, task.expected),
            }
        )

json_correct = sum(result["verified_correct"] for result in json_results)
json_flagged = sum(bool(result["hack_signatures"]) for result in json_results)
print(f"JSON verified-correct rate: {json_correct / len(json_results):.0%} "
      f"({json_correct}/{len(json_results)})")
print(f"Potential exploit or contract-failure signatures: {json_flagged}/{len(json_results)}")
for result in json_results:
    print(f"  reward={result['reward']:.1f} flags={result['hack_signatures']}")
    print(f"  {result['response']}")
print("These flags require review; they are not proof of deliberate reward hacking.")
print("Detection uses duplicate-key rejection, exact keys/types, and independent "
      "recomputation of every derived value; hidden randomized inputs would harden it further.")

JSON verified-correct rate: 56% (5/9)
Potential exploit or contract-failure signatures: 4/9
  reward=1.0 flags=[]
  {"slug": "hello-rlvr-16", "length": 13, "contains_digit": true}
  reward=1.0 flags=[]
  {"slug": "hello-rlvr-16", "length": 13, "contains_digit": true}
  reward=1.0 flags=[]
  {"slug": "hello-rlvr-16", "length": 13, "contains_digit": true}
  reward=1.0 flags=[]
  {"slug": "guardrails-audit-trails", "length": 23, "contains_digit": false}
  reward=0.8 flags=['schema-shaped output with incorrect derived values']
  {"slug": "guardrails-audit-trails", "length": 25, "contains_digit": false}
  reward=1.0 flags=[]
  {
  "slug": "guardrails-audit-trails",
  "length": 23,
  "contains_digit": false
}
  reward=0.8 flags=['schema-shaped output with incorrect derived values']
  {"slug": "dpo-preference-pairs", "length": 22, "contains_digit": false}
  reward=0.8 flags=['schema-shaped output with incorrect derived values']
  {"slug": "dpo-preference-pairs", "length": 22, "contains_digit"

## Breakout Room #2 Summary

- Once you train against a verifier, it *is* the objective — Goodhart's law makes reward-hacking detection and an append-only audit trail (`artifacts/verifier.jsonl`) part of the core pipeline, not an afterthought.
- Code verification generalizes the idea: unit tests yield fractional rewards, and executing untrusted generated code demands sandboxing in anything beyond a demo.
- Verified groups become training data two ways: `{prompt, chosen, rejected}` pairs for DPO-style trainers, or raw group rewards for GRPO — with hack-suspect samples excluded so the next policy doesn't learn to cheat.

Where to go next: feed `artifacts/preferences.jsonl` to TRL's [`DPOTrainer`](https://huggingface.co/docs/trl/dpo_trainer); plug these reward functions into Session 15's GRPO run; or read how the labs do it at scale — [Tülu 3](https://arxiv.org/abs/2411.15124) (which coined RLVR) and [DeepSeek-R1](https://arxiv.org/abs/2501.12948).

---
## Conclusion: What We Built, Start to Finish

Walking back through the notebook, the full RLVR arc was:

1. **A policy** (Task 1) — a small API model sampled at temperature 1.0, chosen precisely because it is *sometimes wrong*.
2. **A verifiable domain** (Task 2) — math problems with ground-truth answers and a `\boxed{}` convention that makes checking mechanical.
3. **A reward function** (Task 3) — deterministic, normalized, and asymmetric (+1.0 / −0.1) so early failure doesn't teach refusal.
4. **The sampling loop** (Task 4) — *groups* of completions per prompt, the same structure GRPO computes advantages over, and the source of correct/incorrect contrast.
5. **Adversarial thinking** (Task 5) — once you train against a verifier it *is* the objective, so hack detection and an append-only audit trail are part of the pipeline, not an afterthought.
6. **A second domain** (Task 6) — code judged by unit tests, showing rewards can be fractional and that verification can mean *executing untrusted output*.
7. **Training data** (Task 7) — audited verifier decisions became `{prompt, chosen, rejected}` pairs, with hack-suspects excluded so the next policy iteration doesn't learn to cheat.

The single idea underneath all of it: **wherever a deterministic program can check correctness, you can turn cheap inference into training signal** — no human labelers, no learned reward model. The verifier's quality *is* the ceiling on the policy's quality.

## What Looks Different in Production

This notebook is the smallest honest version of RLVR. Scaling it into a real training pipeline changes nearly every component:

**Sandboxed code execution.** Our `CodeVerifier` runs model-generated code in a bare `subprocess` on your machine — fine for a demo, unacceptable in production, where the policy *will* eventually generate code that reads the filesystem, opens sockets, or fork-bombs the host (and our timeout catches none of that). Production verifiers execute candidates in isolated, disposable environments: [Vercel Sandbox](https://vercel.com/docs/vercel-sandbox) is a good example — ephemeral microVMs built to run untrusted, LLM-generated code with CPU/memory limits, network controls, and full teardown after each run. Self-hosted equivalents include gVisor, Firecracker microVMs, or locked-down containers. The rule: the verifier defines the reward, so the verifier's execution environment is a *security boundary*.

**Scale and throughput.** Five problems × 4 samples becomes tens of thousands of prompts × 8–64 samples per RL step. Sequential API calls give way to async/batched sampling against a dedicated inference fleet (vLLM, as in Session 15) — generation throughput, not the policy update, is usually the bottleneck.

**Verifier hardening.** Regex extraction and exact match get replaced by symbolic math checkers (e.g., SymPy-based equivalence), multiple test suites with hidden held-out cases, and stacked format rewards — because at scale, every blind spot *will* be found and exploited.

**Governance.** Our `verifier.jsonl` becomes real infrastructure: versioned datasets, per-run ledgers, flagged-sample review queues, and dashboards tracking reward distributions for drift. When someone asks "was this model trained on honest rewards?", the audit trail is the answer — this is exactly the artifact regulated environments (the origin of this material) require.

**The training loop itself.** The preference pairs here feed an actual policy update — DPO or GRPO — and then the loop *repeats* against the updated policy: sample, verify, update, again. One pass through this notebook is a single iteration of that flywheel.